# Motor fault prediction - Random Forest training

Evaluate with each historical machine held out in turn, then fit the deployment model on the historical rows plus any manually verified live labels. Only vibration and temperature are used. The historical score does not guarantee accuracy on the physical ESP32; calibrate sensors and validate against independently confirmed real readings.

In [5]:
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict

from motor_faulty_prediction.train_model import (
    ADC_COUNTS_PER_VIBRATION_UNIT,
    MAX_DEPTH,
    MIN_SAMPLES_LEAF,
    RANDOM_STATE,
    TREE_COUNT,
    generate_header,
)

PROJECT_ROOT = Path.cwd()
DATASET_PATH = PROJECT_ROOT / "predictive_maintenance_dataset.csv"
LIVE_DATASET_PATH = PROJECT_ROOT / "labeled_motor_sensor_readings.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "motor_fault_random_forest.joblib"
METRICS_PATH = PROJECT_ROOT / "models" / "training_metrics.json"

if not DATASET_PATH.is_file():
    raise FileNotFoundError(f"Historical dataset not found: {DATASET_PATH}")

print(f"Project: {PROJECT_ROOT}")
print(f"Dataset: {DATASET_PATH.name}")

ModuleNotFoundError: No module named 'motor_faulty_prediction'

## Load and inspect historical data

In [2]:
data = pd.read_csv(DATASET_PATH, parse_dates=["timestamp"])
required_columns = {"timestamp", "vibration", "temperature", "label"}
missing_columns = required_columns.difference(data.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")

data = data.sort_values("timestamp", kind="stable").reset_index(drop=True)
data[["vibration", "temperature", "label"]] = data[
    ["vibration", "temperature", "label"]
].apply(pd.to_numeric, errors="raise")
if data[["vibration", "temperature", "label"]].isna().any().any():
    raise ValueError("Training columns contain missing values.")
if not set(data["label"].astype(int).unique()) == {0, 1}:
    raise ValueError("The label column must contain both 0 (normal) and 1 (fault).")

FEATURES = ["vibration", "temperature"]
TARGET = "label"
if LIVE_DATASET_PATH.is_file():
    live_data = pd.read_csv(LIVE_DATASET_PATH)
    live_required = {"timestamp", *FEATURES, TARGET}
    missing_live = live_required.difference(live_data.columns)
    if missing_live:
        raise ValueError(f"Labeled live dataset is missing columns: {sorted(missing_live)}")
    live_data[list(live_required.difference({"timestamp"}))] = live_data[
        list(live_required.difference({"timestamp"}))
    ].apply(pd.to_numeric, errors="raise")
    if live_data[list(live_required.difference({"timestamp"}))].isna().any().any():
        raise ValueError("Labeled live dataset contains missing training values.")
    if not set(live_data[TARGET].astype(int).unique()).issubset({0, 1}):
        raise ValueError("Live labels must be 0 (normal) or 1 (fault).")
else:
    live_data = pd.DataFrame(columns=[*FEATURES, TARGET])
display(data[FEATURES + [TARGET]].head())
display(data[TARGET].value_counts().rename(index={0: "normal", 1: "fault"}))
display(data[FEATURES].describe().round(3))
print(f"Historical rows: {len(data)} | labeled live rows: {len(live_data)} | first timestamp: {data.timestamp.min()} | last timestamp: {data.timestamp.max()}")

,vibration,temperature,label
0,0.822,66.85,0
1,0.726,63.28,0
2,0.843,60.66,0
3,1.398,76.20,1
4,0.815,63.28,0


label
normal    1598
fault      202
Name: count, dtype: int64

,vibration,temperature
count,1800.000,1800.000
mean,0.843,66.356
std,0.137,4.448
min,0.645,58.360
25%,0.769,63.730
50%,0.805,65.355
75%,0.851,66.990
max,1.405,85.160


Historical rows: 1800 | labeled live rows: 0 | first timestamp: 2024-07-01 08:00:00 | last timestamp: 2024-07-01 17:59:00


## Evaluate by holding out each machine

Leave-one-machine-out evaluation avoids reporting adjacent rows from the same machine as independent evidence. These metrics use historical data only; confirmed live labels are reserved for the final deployment fit.

In [3]:
X = data[FEATURES].to_numpy(dtype=float)
y = data[TARGET].to_numpy(dtype=int)
machine_groups = data["machine_id"].astype(str).to_numpy()
if len(set(machine_groups)) < 2:
    raise ValueError("Leave-one-machine-out evaluation requires at least two machines.")

evaluation_model = RandomForestClassifier(
    n_estimators=TREE_COUNT,
    max_depth=MAX_DEPTH,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=1,
)
y_pred = cross_val_predict(
    evaluation_model, X, y, groups=machine_groups,
    cv=LeaveOneGroupOut(), n_jobs=1,
)

holdout_accuracy = accuracy_score(y, y_pred)
holdout_confusion = confusion_matrix(y, y_pred, labels=[0, 1])
holdout_report = classification_report(
    y,
    y_pred,
    labels=[0, 1],
    target_names=["normal", "fault"],
    output_dict=True,
    zero_division=0,
)

print(f"Leave-one-machine-out validation: {len(X)} out-of-fold historical rows across {len(set(machine_groups))} machines")
print(f"Cross-validated accuracy: {holdout_accuracy:.2%}")
print("Confusion matrix [normal, fault]:")
display(pd.DataFrame(holdout_confusion, index=["actual normal", "actual fault"], columns=["predicted normal", "predicted fault"]))
display(pd.DataFrame(holdout_report).T.round(3))

Leave-one-machine-out validation: 1800 out-of-fold historical rows across 3 machines
Cross-validated accuracy: 99.89%
Confusion matrix [normal, fault]:


,predicted normal,predicted fault
actual normal,1597,1
actual fault,1,201


,precision,recall,f1-score,support
normal,0.999,0.999,0.999,1598.000
fault,0.995,0.995,0.995,202.000
accuracy,0.999,0.999,0.999,0.999
macro avg,0.997,0.997,0.997,1800.000
weighted avg,0.999,0.999,0.999,1800.000


## Fit on historical and confirmed live rows; export for the ESP32

After historical cross-validation, fit the deployment model on all historical rows and any manually verified live labels. The generated header contains the same forest for on-device inference.

In [4]:
final_model = RandomForestClassifier(
    n_estimators=TREE_COUNT,
    max_depth=MAX_DEPTH,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=1,
)
X_deploy = pd.concat([data[FEATURES], live_data[FEATURES]], ignore_index=True).to_numpy(dtype=float)
y_deploy = pd.concat([data[TARGET], live_data[TARGET]], ignore_index=True).to_numpy(dtype=int)
final_model.fit(X_deploy, y_deploy)

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(final_model, MODEL_PATH)
generate_header(final_model, X_deploy.tolist())

metrics = {
    "algorithm": "RandomForestClassifier",
    "features": FEATURES,
    "historical_samples": int(len(data)),
    "labeled_live_samples": int(len(live_data)),
    "final_training_samples": int(len(X_deploy)),
    "validation_samples": int(len(X)),
    "validation_groups": sorted(set(machine_groups)),
    "validation_strategy": "leave-one-machine-out cross-validation",
    "fault_samples": int(y_deploy.sum()),
    "normal_samples": int(len(y_deploy) - y_deploy.sum()),
    "trees": TREE_COUNT,
    "max_depth": MAX_DEPTH,
    "min_samples_leaf": MIN_SAMPLES_LEAF,
    "random_state": RANDOM_STATE,
    "accuracy": float(holdout_accuracy),
    "confusion_matrix": holdout_confusion.tolist(),
    "classification_report": holdout_report,
    "feature_ranges": {
        feature: [float(pd.concat([data[feature], live_data[feature]], ignore_index=True).min()), float(pd.concat([data[feature], live_data[feature]], ignore_index=True).max())]
        for feature in FEATURES
    },
    "adc_counts_per_vibration_unit": ADC_COUNTS_PER_VIBRATION_UNIT,
}
METRICS_PATH.write_text(json.dumps(metrics, indent=2) + "\n", encoding="utf-8")

print(f"Saved Random Forest: {MODEL_PATH}")
print(f"Saved ESP32 forest header: {PROJECT_ROOT / 'src' / 'motor_faulty_prediction' / 'embedded_model.h'}")
print(f"Saved cross-validation report: {METRICS_PATH}")
print(f"Cross-validated historical accuracy: {holdout_accuracy:.2%}; labeled live samples added: {len(live_data)}")
print("Warning: live vibration conversion must be calibrated against the physical ADXL335.")

Saved Random Forest: C:\Users\NITHISH\OneDrive\Desktop\motor faulty prediction\models\motor_fault_random_forest.joblib
Saved ESP32 forest header: C:\Users\NITHISH\OneDrive\Desktop\motor faulty prediction\src\motor_faulty_prediction\embedded_model.h
Saved cross-validation report: C:\Users\NITHISH\OneDrive\Desktop\motor faulty prediction\models\training_metrics.json
Cross-validated historical accuracy: 99.89%; labeled live samples added: 0
